# 📝 Hindi Legal Event Annotation — Auto-Fill "Not Clearly Specified" Fields

This notebook reads each Hindi court judgment and uses the Claude API to fill in fields marked as **"Not Clearly Specified"**.

### Setup
1. Install: `pip install anthropic pandas`
2. Set your Anthropic API key below
3. Place `hindi_not_clearly_specified_rows.csv` in the same folder
4. Run all cells — output saved as `hindi_annotated_output.csv`


In [1]:
!pip install anthropic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 5.3 MB/s  0:00:00m eta 0:00:01
    torch (>=1.7.*)
           ~~~~~~^


In [ ]:
# Install required packages (run once)
# !pip install anthropic pandas

import pandas as pd
import json
import time
import os
from anthropic import Anthropic

# ========================================
# SET YOUR API KEY HERE
# ========================================
API_KEY = "sk-ant-..."  # <-- Paste your Anthropic API key

client = Anthropic(api_key=API_KEY)
print("✅ Anthropic client initialized")


In [ ]:
# Load the dataset
INPUT_FILE = "hindi_not_clearly_specified_rows.csv"
OUTPUT_FILE = "hindi_annotated_output.csv"
PROGRESS_FILE = "annotation_progress.json"

df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
df.columns = ['case_id','language','case_type','judgment_text','subject','object','objective_aspect','subjective_aspect','reasoning']

# Identify rows needing processing
def has_ncs(row):
    for col in ['subject','object','objective_aspect','subjective_aspect']:
        if 'Not Clearly Specified' in str(row[col]):
            return True
    return False

ncs_rows = df[df.apply(has_ncs, axis=1)]
print(f"Total rows: {len(df)}")
print(f"Rows with 'Not Clearly Specified': {len(ncs_rows)}")

# Count NCS per column
for col in ['subject','object','objective_aspect','subjective_aspect']:
    n = df[col].astype(str).str.contains('Not Clearly Specified', case=False).sum()
    print(f"  {col}: {n} NCS cells")


In [ ]:
SYSTEM_PROMPT = """You are an expert Legal Event Annotator for Indian court judgments.

You will receive a Hindi court judgment text with its case_type and current annotations.
Your task: For any field containing "Not Clearly Specified", read the ENTIRE judgment and extract the correct value.

DEFINITIONS BY CASE TYPE:

CRIMINAL:
- subject = Accused (person alleged to have committed the offence)
- object = Victim (person/property/interest harmed)
- objective_aspect = Criminal Act (murder, theft, cheating, etc. — NOT IPC sections)
- subjective_aspect = Criminal Intent/Mens Rea (intention to kill, dishonest intention, etc.)

CIVIL:
- subject = Defendant
- object = Plaintiff
- objective_aspect = Cause of Action (breach of contract, property dispute, etc.)
- subjective_aspect = Remedy Sought (compensation, injunction, declaration, etc.)

CONSTITUTIONAL:
- subject = State Respondent (government authority challenged)
- object = Petitioner
- objective_aspect = Constitutional Breach (violation of Article 14/19/21, illegal detention, etc.)
- subjective_aspect = Writ Relief (Mandamus, Certiorari, Habeas Corpus, etc.)

ADMINISTRATIVE:
- subject = Administrative Authority (government department challenged)
- object = Aggrieved Party
- objective_aspect = Impugned Action (the administrative action under challenge)
- subjective_aspect = Grounds for Review (arbitrariness, illegality, procedural unfairness, etc.)

CRITICAL RULES:
1. Use ONLY information explicitly in the judgment text
2. Never invent or guess — accuracy > completeness
3. If truly not identifiable, keep "Not Specified"
4. Multiple parties: separate with semicolons
5. For objective_aspect in criminal cases, write the OFFENCE NAME, not IPC section numbers

Respond ONLY with a JSON object containing all four fields:
{"subject": "...", "object": "...", "objective_aspect": "...", "subjective_aspect": "..."}

For fields that already have valid values (not "Not Clearly Specified"), return them UNCHANGED.
"""
print("✅ System prompt ready")


In [ ]:
def process_judgment(row, client):
    """Send one judgment to Claude API and get filled annotations."""
    judgment = str(row['judgment_text'])
    
    # Truncate very long texts (keep beginning and end for context)
    if len(judgment) > 20000:
        judgment = judgment[:10000] + "\n...\n[MIDDLE SECTION TRUNCATED FOR LENGTH]\n...\n" + judgment[-10000:]
    
    # Identify which fields need filling
    ncs_fields = []
    for col in ['subject','object','objective_aspect','subjective_aspect']:
        if 'Not Clearly Specified' in str(row[col]):
            ncs_fields.append(col)
    
    user_msg = f"""Case Type: {row['case_type']}

Current Annotations:
- subject: {row['subject']}
- object: {row['object']}
- objective_aspect: {row['objective_aspect']}
- subjective_aspect: {row['subjective_aspect']}

Fields to fill: {ncs_fields}

--- JUDGMENT TEXT ---
{judgment}
--- END ---

Extract and fill the "Not Clearly Specified" fields. Return JSON only."""

    try:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1000,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}]
        )
        
        text = ""
        for block in response.content:
            if block.type == "text":
                text += block.text
        
        text = text.strip()
        # Remove markdown fences
        if text.startswith("```"):
            text = text.split("\n", 1)[1] if "\n" in text else text[3:]
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()
        
        result = json.loads(text)
        return result
    
    except json.JSONDecodeError:
        # Try to find JSON in text
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0 and end > start:
            try:
                return json.loads(text[start:end])
            except:
                pass
        print(f"    ⚠️ JSON parse error: {text[:150]}")
        return None
    
    except Exception as e:
        print(f"    ❌ API error: {e}")
        return None

print("✅ Processing function ready")


In [ ]:
# Load progress (allows resuming if interrupted)
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
    print(f"📂 Resuming — {len(progress)} rows already processed")
else:
    progress = {}

# Process all rows
total = len(df)
processed = 0
errors = 0
skipped = 0

for idx, row in df.iterrows():
    case_id = str(row['case_id'])
    
    # Skip already processed
    if case_id in progress:
        if progress[case_id].get('status') == 'done':
            # Apply saved results
            result = progress[case_id].get('result', {})
            for field in ['subject','object','objective_aspect','subjective_aspect']:
                if field in result and 'Not Clearly Specified' in str(row[field]):
                    df.at[idx, field] = result[field]
        continue
    
    # Skip rows without NCS
    if not has_ncs(row):
        progress[case_id] = {"status": "no_ncs"}
        skipped += 1
        continue
    
    ncs_fields = [c for c in ['subject','object','objective_aspect','subjective_aspect']
                  if 'Not Clearly Specified' in str(row[c])]
    
    print(f"[{idx+1}/{total}] {case_id} ({row['case_type']}) — filling: {ncs_fields}", end=" ")
    
    result = process_judgment(row, client)
    
    if result:
        progress[case_id] = {"status": "done", "result": result}
        for field in ['subject','object','objective_aspect','subjective_aspect']:
            if field in result and 'Not Clearly Specified' in str(row[field]):
                df.at[idx, field] = result[field]
        processed += 1
        print("✅")
    else:
        progress[case_id] = {"status": "error"}
        errors += 1
        print("❌")
    
    # Save progress every 5 rows
    if (processed + errors) % 5 == 0:
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(progress, f, ensure_ascii=False, indent=2)
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    # Rate limit: ~0.5s delay
    time.sleep(0.5)

# Final save
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
with open(PROGRESS_FILE, 'w') as f:
    json.dump(progress, f, ensure_ascii=False, indent=2)

print(f"\n{'='*50}")
print(f"✅ COMPLETE: {processed} filled, {skipped} skipped (no NCS), {errors} errors")
print(f"📄 Output saved to: {OUTPUT_FILE}")


## Verify Results

In [ ]:
# Load and verify output
df_out = pd.read_csv(OUTPUT_FILE, encoding='utf-8-sig')

print("=== Before vs After: 'Not Clearly Specified' counts ===\n")

df_orig = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
df_orig.columns = ['case_id','language','case_type','judgment_text','subject','object','objective_aspect','subjective_aspect','reasoning']

for col in ['subject','object','objective_aspect','subjective_aspect']:
    before = df_orig[col].astype(str).str.contains('Not Clearly Specified', case=False).sum()
    after = df_out[col].astype(str).str.contains('Not Clearly Specified', case=False).sum()
    filled = before - after
    print(f"{col}:")
    print(f"  Before: {before} NCS → After: {after} NCS  ({filled} filled)")

print(f"\n--- Sample filled values ---")
for idx, row in df_out.iterrows():
    orig_row = df_orig.iloc[idx]
    changes = []
    for col in ['subject','object','objective_aspect','subjective_aspect']:
        if 'Not Clearly Specified' in str(orig_row[col]) and 'Not Clearly Specified' not in str(row[col]):
            changes.append(f"  {col}: {str(row[col])[:100]}")
    if changes:
        print(f"\n{row['case_id']} ({row['case_type']}):")
        for c in changes:
            print(c)
    if idx > 15:
        print("\n... (showing first 16 rows)")
        break


In [ ]:
# Final statistics
print("=== Final Dataset Statistics ===\n")
print(f"Total rows: {len(df_out)}")
print(f"\nRemaining 'Not Clearly Specified' (could not be extracted from judgment):")

remaining = 0
for col in ['subject','object','objective_aspect','subjective_aspect']:
    n = df_out[col].astype(str).str.contains('Not Clearly Specified|Not Specified', case=False).sum()
    remaining += n
    print(f"  {col}: {n}")

print(f"\nTotal remaining unspecified: {remaining}")
print(f"\nNote: Some fields remain 'Not Specified' because the judgment")
print(f"genuinely does not contain that information (e.g., short procedural orders).")
